# Observability around wind jumps and odor contact: cosyne logic on the tamagotchi force pipeline

Re-creation of `observability_visualize_cosyne.ipynb` for the data the new pipeline produces:

- eval logs: `tamagotchi.evalCli` (force/dynamics action model, 3-dim actions `T_par, T_perp, tau`)
- observability: `python3 -u -m tamagotchi.eval.observability --log_fname <eval>.pkl --window 4`
  (`<eval>_w4_observability.pkl`, one `[EV_df, t_sim, x_sim, window, ep_idx]` per episode)

What is kept from the cosyne notebook: drop the last row of every episode, merge EV with the
trajectory on `(ep_idx, time)`, shift `zeta` by `window/2` and drop the first `window` rows of
every episode, align to wind changes / odor contacts with the "alternative trial filtering"
rules, median +- SEM per aligned time index, `colorline` coloured by `zeta_shifted` on a log
scale, a dashed null line (whole-dataset median +- SEM), shared y limits between the wind and
odor figures, and a separate colourbar figure.

What changed and why:

- dt is hard-coded to `DT = 0.1` (the value `tamagotchi.eval.observability.DT` simulates with);
  every window/guard that the old notebook wrote in steps at 0.04 s is now given in seconds and
  converted with `steps()`.
- the plotted quantities are *true kinematic quantities* instead of action-derived ones:
  `speed_dt_abs` = |d(airspeed)/dt| [m/s^2] and `ang_vel_dt_abs` = |d(angular velocity)/dt| [rad/s^2],
  where airspeed = |info['air_velcity']| and angular velocity = info['ang_vel'] (force physics).
  No `*2` / `*6*pi` action scaling.
- `noisy_jitter_coverage` wind is Poisson direction jumps (N(0, 25 deg) every ~3 s) on top of
  an OU jitter (3 deg, tau 1 s), so the wind direction changes every step. A "wind change" is now
  a step whose |delta wind angle| exceeds `wind_jump_thresh_deg` (10 deg) instead of any non-zero
  difference (`log_analysis.calc_time_since_last_wind_change`).
- the colouring variable is only `zeta_shifted`.

Added after the aligned figures, to see which data the traces come from:

- **event windows in space**: every +-window around a wind jump / odor contact, all episodes
  rotated back by `-rotate_by` (from the per-step `info`) so all plume rotations align with 0 deg,
  drawn with `draw_window_panel` of `tamagotchi/eval/outcome_start_plots.py` (branch
  `claude/jolly-cray-lsu84u`), once with *all* windows and once with only the windows that pass
  the eligibility rules of the aligned figures.
- **sensory trace collections** (`_preprint_observability.ipynb`): per-window traces of the wind
  direction / odor, the two kinematic quantities and `zeta_shifted` against time since the event,
  again for all and for eligible windows.

Run from a directory under `/gscratch/portia/...` (tamagotchi.config picks `datadir` from the cwd).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.collections as mcoll
import tamagotchi.eval.log_analysis as log_analysis
from tamagotchi import config
from pybounds import colorline
# branch claude/jolly-cray-lsu84u of tamagotchi (draw_window_panel, _limits); the module switches
# matplotlib to the Agg backend on import, the next cell switches back to inline
from tamagotchi.eval.outcome_start_plots import draw_window_panel, _rotate, _limits, PANEL_STYLE

DT = 0.1  # [s] hard-coded; must equal env_dt of the run and tamagotchi.eval.observability.DT


def steps(seconds):
    """Seconds -> whole env steps at DT."""
    return int(round(seconds / DT))


def calc_time_since_last_wind_jump(eps_df, jump_thresh_deg=10.0, dt=DT):
    """Time since the last wind-direction *jump* for one episode.

    Replacement for log_analysis.calc_time_since_last_wind_change: with the jittered wind of
    noisy_jitter_coverage the direction differs at every step, so only steps whose |delta theta|
    exceeds `jump_thresh_deg` count as a change. Returns a copy with
    `wind_dtheta_deg`, `wind_jump` (0/1) and `time_since_last_wind_change` (0 at a jump, `dt`
    one step later, ...), the same convention as the old function.
    """
    theta = np.unwrap(np.arctan2(eps_df['wind_angle_ground_y'].values.astype(float),
                                 eps_df['wind_angle_ground_x'].values.astype(float)))
    dtheta_deg = np.rad2deg(np.diff(theta, prepend=theta[0]))
    jump = np.abs(dtheta_deg) > jump_thresh_deg
    jump[0] = False
    since = np.zeros(len(eps_df))
    count = 0.0
    for i in range(len(eps_df)):
        count += dt
        if jump[i]:
            count = 0.0
        since[i] = count
    out = eps_df.copy()
    out['wind_dtheta_deg'] = dtheta_deg
    out['wind_jump'] = jump.astype(int)
    out['time_since_last_wind_change'] = since
    return out


def collect_windows(df_traj, event_col, around_range, eligible_fn):
    """Every +-around_range window around the rows where `event_col` == 1, per episode.

    Returns the concatenated windows (clipped to the episode) with `plot_tidx` (0 at the event),
    `event_idx` (one id per window) and `eligible`: the window is complete and `eligible_fn(ep_df,
    event_idx, lower_bound, upper_bound)` holds. The aligned figures use the eligible windows only;
    the "all windows" figures use everything.
    """
    windows = []
    for _, ep_df in df_traj.groupby('ep_idx', sort=False):
        for ev_idx in ep_df.index[ep_df[event_col] == 1]:
            lower_bound, upper_bound = ev_idx - around_range, ev_idx + around_range
            complete = lower_bound >= ep_df.index.min() and upper_bound < ep_df.index.max()  # cosyne rule
            ok = bool(complete and eligible_fn(ep_df, ev_idx, lower_bound, upper_bound))
            lo = max(lower_bound, ep_df.index.min())
            hi = min(upper_bound, ep_df.index.max())
            w = ep_df.loc[lo:hi].copy()  # index labels are contiguous within an episode
            w['plot_tidx'] = w.index - ev_idx
            w['event_idx'] = len(windows)
            w['eligible'] = ok
            windows.append(w)
    if not windows:
        return df_traj.iloc[0:0].assign(plot_tidx=pd.Series(dtype=int), event_idx=pd.Series(dtype=int),
                                        eligible=pd.Series(dtype=bool))
    return pd.concat(windows)


def tidx_around_wind_change(df_traj, around_range, stable_guard, min_time_from_init=1.0,
                            min_stable_before=1.0, drop_close_to_trial_init=True, verbose=False):
    """All +-around_range windows around wind jumps (`wind_jump` == 1), with `eligible` set by the
    "alternative trial filtering" of the cosyne notebook: the window is complete, its start is
    >= `min_time_from_init` s into the episode (if drop_close_to_trial_init), the wind was stable
    for >= `min_stable_before` s at the window start, and no other jump lies within
    +-`stable_guard` steps of the jump.
    """
    def eligible(ep_df, ev_idx, lower_bound, upper_bound):
        if drop_close_to_trial_init and ep_df['time'].loc[lower_bound] < min_time_from_init:
            return False
        if ep_df.loc[ev_idx + 1:ev_idx + stable_guard, 'wind_jump'].eq(1).any():
            return False
        if ep_df.loc[ev_idx - stable_guard:ev_idx - 1, 'wind_jump'].eq(1).any():
            return False
        if ep_df['time_since_last_wind_change'].loc[lower_bound] < min_stable_before:
            return False
        return True

    windows = collect_windows(df_traj, 'wind_jump', around_range, eligible)
    n_all = windows['event_idx'].nunique()
    n_ok = windows.loc[windows['eligible'], 'event_idx'].nunique()
    print(f"[NOTE]: number of eligible unique wind change instances {n_ok} (of {n_all} windows)")
    if verbose:
        ok = windows[windows['eligible']]
        print(f"eligible rows {ok.shape} with {ok['ep_idx'].nunique()} eps; "
              f"eps {set(df_traj['ep_idx'].unique()) - set(ok['ep_idx'].unique())} had no eligible wind change window")
    return windows


def mark_odor_contact_start(df_traj):
    """0/1 column: first on-plume row of every continuous odor contact (per episode)."""
    prev = df_traj.groupby('ep_idx', sort=False)['odor_01'].shift(1).fillna(0)
    return ((df_traj['odor_01'] == 1) & (prev == 0)).astype(int)


def tidx_around_odor_contact(df_traj, around_range, odor_guard, verbose=False):
    """All +-around_range windows around odor contact starts (`odor_contact_start` == 1), with
    `eligible` set by the "alternative trial filtering" of the cosyne notebook: the window is
    complete, the `odor_guard` steps from the contact on are all on plume, the `odor_guard` steps
    before it are all off plume, and (redundantly for guard >= range) the window is off then on.
    """
    def eligible(ep_df, ev_idx, lower_bound, upper_bound):
        if ep_df.loc[ev_idx:ev_idx + odor_guard, 'odor_01'].eq(0).any():
            return False
        if ep_df.loc[ev_idx - odor_guard:ev_idx - 1, 'odor_01'].eq(1).any():
            return False
        if ep_df.loc[lower_bound:ev_idx - 1, 'odor_01'].eq(1).any():
            return False
        if ep_df.loc[ev_idx:upper_bound, 'odor_01'].eq(0).any():
            return False
        return True

    windows = collect_windows(df_traj, 'odor_contact_start', around_range, eligible)
    n_all = windows['event_idx'].nunique()
    n_ok = windows.loc[windows['eligible'], 'event_idx'].nunique()
    print(f"[NOTE]: number of eligible unique odor contact instances {n_ok} (of {n_all} windows)")
    if verbose:
        ok = windows[windows['eligible']]
        print(f"eligible rows {ok.shape} with {ok['ep_idx'].nunique()} eps; "
              f"eps {set(df_traj['ep_idx'].unique()) - set(ok['ep_idx'].unique())} had no eligible odor contact window")
    return windows


def colorline_trace(x, y, z, ax=None, cmap='inferno_r', norm=None, linewidth=1.5, alpha=1.0):
    """colorline with flat caps (no overlapping round-cap dots when alpha < 1); _preprint_observability.ipynb."""
    z = np.asarray(z)
    if norm is None:
        norm = plt.Normalize(np.nanmin(z), np.nanmax(z))
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcoll.LineCollection(segments, array=z, cmap=cmap, norm=norm, linewidth=linewidth, alpha=alpha,
                              capstyle='butt')
    if ax is None:
        ax = plt.gca()
    ax.add_collection(lc)
    return lc

In [ ]:
%matplotlib inline
# (outcome_start_plots sets matplotlib.use("Agg") on import; restore inline figures)

## spec plot what

In [ ]:
plot_around_odor = True
plot_around_wind = True
filter_null = False      # null baseline from all eps (False) or only eps with eligible windows (True)
median = True            # median (True) or mean (False) per aligned time index and for the null line
drop_close_to_trial_init = True
save = False
titles_on = False

# windows and guards in seconds (old notebook: steps at 0.04 s -> 25, 15, 18, 20 steps)
wind_around_s = 1.0        # +- window around a wind jump
odor_around_s = 0.6        # +- window around an odor contact start
wind_stable_guard_s = 0.72 # no other wind jump within +- this of the jump
odor_guard_s = 0.8         # all-odor after / no-odor before the contact for this long
min_time_from_init_s = 1.0 # window start must be this far into the episode
min_stable_before_s = 1.0  # wind stable for this long at the window start
wind_jump_thresh_deg = 10.0  # |delta wind angle| per step above this = Poisson jump (OU jitter is ~3 deg)
plot_xlim_s = 0.6

wind_around_range = steps(wind_around_s)
odor_around_range = steps(odor_around_s)
print(f"DT {DT}s: wind around range {wind_around_range} steps ({wind_around_range * DT:.2f}s), "
      f"odor around range {odor_around_range} steps ({odor_around_range * DT:.2f}s), "
      f"wind guard {steps(wind_stable_guard_s)} steps, odor guard {steps(odor_guard_s)} steps")

# true kinematic quantities (airspeed from info air_velcity, angular velocity from info ang_vel)
ylabl_title = {'speed': 'airspeed\n(m/s)',
               'speed_dt': 'acceleration\n(m/s$^2$)',
               'speed_dt_abs': 'absolute\nacceleration\n(m/s$^2$)',
               'ang_vel': 'angular\nvelocity\n(rad/s)',
               'ang_vel_abs': 'angular\nspeed\n(rad/s)',
               'ang_vel_dt': 'angular\nacceleration\n(rad/s$^2$)',
               'ang_vel_dt_abs': 'absolute\nangular\nacceleration\n(rad/s$^2$)'}
kin_cols = list(ylabl_title.keys())

states = ['speed_dt_abs', 'ang_vel_dt_abs']
n_state = len(states)
fig_size = (10, n_state * 4)

# Plot!
## load datasets

In [ ]:
import importlib 
importlib.reload(log_analysis)  # reload to get the new rotate_by column in get_traj_and_activity_and_stack_them
tamagotchi_root = '/gscratch/portia/jqhu/work/active_sensing/tamagotchi'
window = 4  # --window of tamagotchi.eval.observability
eval_pkls = [
    f'{tamagotchi_root}/data/wind_sensing/apparent_wind_visual_feedback/CL_sweep/eval/plume_seed-1-f9b1fbb5_stage_afb487d7.chkpt/eval_noisy_jitter_coveragex5b5.pkl',
]
obs_pkls = [p.replace('.pkl', f'_w{window}_observability.pkl') for p in eval_pkls]
n_eps_max = 1000  # pull all episodes (HOME and non-HOME)

ls_EV_no_nan_shifted = []
ls_model_name = []
ls_outdir = []
for agent_i, (eval_pkl, obs_pkl) in enumerate(zip(eval_pkls, obs_pkls)):
    eval_folder = os.path.dirname(eval_pkl) + '/'
    dataset = os.path.basename(eval_pkl).replace('.pkl', '')
    model_name = os.path.basename(os.path.dirname(eval_pkl))  # checkpoint stem
    print(f"now visualizing {model_name}, {dataset}")
    if not os.path.exists(obs_pkl):
        raise FileNotFoundError(f"Observability file {obs_pkl} not found")

    with open(obs_pkl, 'rb') as f_handle:
        observability_tupl = pickle.load(f_handle)
    print(f"Found {len(observability_tupl)} trials in observability results")

    selected_df = log_analysis.get_selected_df(eval_folder, [dataset],
                                               n_episodes_home=n_eps_max,
                                               n_episodes_other=n_eps_max,
                                               balanced=False,
                                               oob_only=False,
                                               verbose=False,
                                               log_fname=eval_pkl)
    traj_df_stacked, _ = log_analysis.get_traj_and_activity_and_stack_them(selected_df,
                                                                          obtain_neural_activity=False,
                                                                          obtain_traj_df=True,
                                                                          extended_metadata=False)  # adds 'time' (round 2)
    print(traj_df_stacked.shape)

    ls_EV_no_nan, ls_t_sim, ls_x_sim, ls_window_size, ls_eps_idx = zip(*observability_tupl)
    window_sizes = set(ls_window_size)
    assert len(window_sizes) == 1, f"mixed window sizes {window_sizes}"
    window_size = window_sizes.pop()
    print("window_size", window_size, f"({window_size * DT:.2f}s)")

    # for every episode, drop the last row: there's no terminal+1 state (t_sim has T-1 entries)
    traj_df_stacked = traj_df_stacked.reset_index(drop=True)
    last_rows = traj_df_stacked.groupby('ep_idx').tail(1).index
    print('dropping', len(last_rows), 'rows, which are the last rows of each episode')
    filtered_df = traj_df_stacked.drop(index=last_rows).reset_index(drop=True)

    # plume rotation of each episode (--eval_rotations); 0 when the info has none
    ep_rotate_by = {row['idx']: float(row['log']['infos'][0][0].get('rotate_by') or 0.0)
                    for _, row in selected_df.iterrows()}
    filtered_df['rotate_by'] = filtered_df['ep_idx'].map(ep_rotate_by)
    print("rotate_by counts (episodes):", pd.Series(ep_rotate_by).value_counts().to_dict())
    # back to the unrotated plume frame (rotate by -rotate_by, as outcome_start_plots.episode_frame)
    xy = _rotate(filtered_df[['loc_x', 'loc_y']].values.astype(float), -filtered_df['rotate_by'].values)
    filtered_df['loc_x_unrot'], filtered_df['loc_y_unrot'] = xy[:, 0], xy[:, 1]
    wind_xy = _rotate(filtered_df[['wind_angle_ground_x', 'wind_angle_ground_y']].values.astype(float),
                      -filtered_df['rotate_by'].values)
    filtered_df['wind_zeta_deg_unrot'] = np.rad2deg(np.arctan2(wind_xy[:, 1], wind_xy[:, 0]))

    # true kinematic quantities
    filtered_df['speed'] = np.linalg.norm(np.stack(filtered_df['air_velocity'].values), axis=1)  # airspeed [m/s]
    filtered_df['ground_speed'] = np.linalg.norm(np.stack(filtered_df['allo_ground_velocity'].values), axis=1)
    if 'angular_velocity' in filtered_df.columns:  # force physics: env state ang_vel [rad/s]
        filtered_df['ang_vel'] = filtered_df['angular_velocity'].astype(float)
    else:  # kinematics logs: heading difference per step
        heading = np.angle(filtered_df['agent_angle_x'] + 1j * filtered_df['agent_angle_y'], deg=False)
        filtered_df['heading_phi_unwrap'] = np.concatenate(
            [np.unwrap(heading[idx]) for idx in filtered_df.groupby('ep_idx', sort=False).indices.values()])
        filtered_df['ang_vel'] = filtered_df.groupby('ep_idx')['heading_phi_unwrap'].diff() / DT
    ep_group = filtered_df.groupby('ep_idx')
    filtered_df['speed_dt'] = ep_group['speed'].diff() / DT              # [m/s^2]
    filtered_df['speed_dt_abs'] = filtered_df['speed_dt'].abs()
    filtered_df['ang_vel_abs'] = filtered_df['ang_vel'].abs()
    filtered_df['ang_vel_dt'] = ep_group['ang_vel'].diff() / DT          # [rad/s^2]
    filtered_df['ang_vel_dt_abs'] = filtered_df['ang_vel_dt'].abs()

    # time since last wind *jump* (threshold on |delta theta|, see calc_time_since_last_wind_jump)
    filtered_df = pd.concat([calc_time_since_last_wind_jump(ep_df, jump_thresh_deg=wind_jump_thresh_deg)
                             for _, ep_df in filtered_df.groupby('ep_idx', sort=False)]).reset_index(drop=True)
    print("filtered_df shape", filtered_df.shape,
          f"| wind jumps: {filtered_df['wind_jump'].sum()} in {filtered_df['ep_idx'].nunique()} eps "
          f"({filtered_df['wind_jump'].mean() * 100:.2f}% of steps)")

    # Preprocess the EV data: stack and merge with filtered_df on (ep_idx, time)
    EV_no_nan = pd.concat([df.assign(ep_idx=ep_idx) for df, ep_idx in zip(ls_EV_no_nan, ls_eps_idx)],
                          ignore_index=True)
    print("EV shape", EV_no_nan.shape, "columns", list(EV_no_nan.columns))
    EV_no_nan['time'] = EV_no_nan['time'].round(2)
    EV_no_nan = EV_no_nan.merge(filtered_df[['ep_idx', 'time', 'time_since_last_wind_change', 'wind_jump',
                                             'wind_dtheta_deg', 'odor_lastenc', 'odor_01', 'odor_eps_log',
                                             'ground_speed', *kin_cols, 'loc_x', 'loc_y', 'loc_x_unrot',
                                             'loc_y_unrot', 'rotate_by', 'wind_angle_ground_theta',
                                             'wind_zeta_deg_unrot', 'outcome']],
                                on=['ep_idx', 'time'], how='inner')
    print(f"{EV_no_nan.shape} shape after merge")

    ##########################################################################################
    ######                         SHIFT DATAFRAME BY EV WINDOW                         ######
    ##########################################################################################
    # NOTE: pybounds already reports EV at 'time' = time_initial + window/2; this extra shift is
    # kept as in the cosyne notebook so the figures are comparable.
    window_length = window_size
    EV_no_nan_shifted = EV_no_nan.copy()
    EV_no_nan_shifted['zeta_shifted'] = EV_no_nan_shifted.groupby('ep_idx')['zeta'].shift(int(window_length / 2))
    # drop the first `window_length` rows of each episode
    EV_no_nan_shifted = EV_no_nan_shifted[EV_no_nan_shifted.groupby('ep_idx').cumcount() >= window_length].copy()
    print(f"{EV_no_nan_shifted.shape} shape of EV_no_nan_shifted")

    ls_EV_no_nan_shifted.append(EV_no_nan_shifted)
    ls_model_name.append(model_name)
    ls_outdir.append(os.path.join(eval_folder, 'observability'))

## check the wind-jump threshold
Per-step |delta wind angle| (deg); the OU jitter sits below a few degrees, the Poisson jumps above.

In [ ]:
fig_chk, ax_chk = plt.subplots(1, 2, figsize=(10, 3.5), dpi=120)
for agent_i, EV_no_nan_shifted in enumerate(ls_EV_no_nan_shifted):
    d = EV_no_nan_shifted['wind_dtheta_deg'].abs()
    ax_chk[0].hist(d, bins=np.linspace(0, max(60, d.max()), 121), histtype='step', label=ls_model_name[agent_i])
    jumps_per_ep = EV_no_nan_shifted.groupby('ep_idx')['wind_jump'].sum()
    ax_chk[1].hist(jumps_per_ep, bins=np.arange(-0.5, jumps_per_ep.max() + 1.5), histtype='step')
    print(f"{ls_model_name[agent_i]}: {(d > wind_jump_thresh_deg).mean() * 100:.2f}% of steps above "
          f"{wind_jump_thresh_deg} deg; jumps per ep median {jumps_per_ep.median()}")
ax_chk[0].axvline(wind_jump_thresh_deg, color='red', linestyle='--', label='threshold')
ax_chk[0].set_yscale('log')
ax_chk[0].set_xlabel('|delta wind angle| per step (deg)')
ax_chk[0].set_ylabel('steps')
ax_chk[0].legend(fontsize=7)
ax_chk[1].set_xlabel('wind jumps per episode')
ax_chk[1].set_ylabel('episodes')
plt.tight_layout()
plt.show()

## plot

In [ ]:
def null_baseline(EV_df, cols, median=True):
    """Whole-dataset central value and SEM per kinematic column (dashed line and blue band)."""
    stat = EV_df[cols].median() if median else EV_df[cols].mean()
    sem = EV_df[cols].sem()
    null_df = pd.DataFrame({'value': stat})
    for c in cols:
        null_df.loc['ste_' + c, 'value'] = sem[c]
    return null_df


def aligned_stats(df_aligned, cols, color_by='zeta_shifted', median=True):
    """Central value and SEM of `cols` and `color_by` per plot_tidx; plot_tidx converted to seconds."""
    g = df_aligned.groupby('plot_tidx')[[color_by, *cols]]
    subset_df = (g.median() if median else g.mean()).reset_index()
    sem_df = g.sem().reset_index()
    for c in [color_by, *cols]:
        subset_df['ste_' + c] = sem_df[c]
    subset_df['plot_tidx'] = subset_df['plot_tidx'] * DT
    return subset_df


def plot_aligned(subset_df, null_df, states, xlabel, cnorm=None, cmap='inferno_r', color_by='zeta_shifted',
                 fig_size=(10, 8), titles_on=False, ylabl_title=None):
    """One column of colorline panels (one per state) coloured by `color_by`, SEM band, null line."""
    if cnorm is None:
        max_ev = np.max(subset_df[color_by].values)
        min_ev = np.min(subset_df[color_by].values)
        log_tick_high = int(np.ceil(np.log10(max_ev)))
        log_tick_low = int(np.floor(np.log10(min_ev)))
        print(f"max_ev {max_ev}, min_ev {min_ev}, log_tick_high {log_tick_high}, log_tick_low {log_tick_low}")
        cnorm = mpl.colors.LogNorm(10 ** log_tick_low, 10 ** log_tick_high)
    log_tick_low, log_tick_high = int(np.log10(cnorm.vmin)), int(np.log10(cnorm.vmax))

    fig, ax = plt.subplots(len(states), 1, figsize=fig_size, dpi=300)
    ax = np.atleast_2d(ax).reshape(-1, 1)
    for n, state_name in enumerate(states):
        dispersion_col = 'ste_' + state_name
        colorline(subset_df['plot_tidx'], subset_df[state_name].values, subset_df[color_by].values,  # x, y, color
                  ax=ax[n, 0], cmap=cmap, norm=cnorm, linewidth=6)
        ax[n, 0].fill_between(subset_df['plot_tidx'],
                              subset_df[state_name].values - subset_df[dispersion_col],
                              subset_df[state_name].values + subset_df[dispersion_col],
                              alpha=0.2, color='grey', edgecolor='black', linewidth=4)
        if titles_on:
            if n == 1:
                cax = ax[n, -1].inset_axes([1.03, -2.5, 0.04, 4.0])
                cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax,
                                    ticks=np.logspace(log_tick_low, log_tick_high, log_tick_high - log_tick_low + 1))
                cbar.set_label('Minimum error variance of wind prediction\n' + r'(rad$^-$$^2$)', rotation=270, fontsize=20, labelpad=15)
                cbar.ax.tick_params(labelsize=20)
            ax[n, 0].set_ylabel(ylabl_title[state_name], fontsize=20)
        # null line and null SEM
        ax[n, 0].axhline(null_df.loc[state_name, 'value'], color='black', linestyle='--', linewidth=4)
        ax[n, 0].fill_between(subset_df['plot_tidx'],
                              null_df.loc[state_name, 'value'] - null_df.loc['ste_' + state_name, 'value'],
                              null_df.loc[state_name, 'value'] + null_df.loc['ste_' + state_name, 'value'],
                              alpha=0.2, color='blue', edgecolor='black', linewidth=0.5)
        ax[n, 0].spines['bottom'].set_linewidth(0.5)
        ax[n, 0].spines['left'].set_linewidth(0.5)
        ax[n, 0].spines['top'].set_linewidth(0)
        ax[n, 0].spines['right'].set_linewidth(0)
        ax[n, 0].set_xlim(subset_df['plot_tidx'].min(), subset_df['plot_tidx'].max())
    for a in ax.flat:
        a.tick_params(axis='both', labelsize=24)
    ax[-1, 0].set_xlabel(xlabel, fontsize=32, labelpad=20)
    fig.subplots_adjust(left=0.15, bottom=0.2, right=0.85, top=0.9, wspace=0.5, hspace=0.4)
    return fig, ax, cnorm


verbose = False
color_by = 'zeta_shifted'
cmap = 'inferno_r'
ls_wind_windows, ls_odor_windows, ls_cnorm = [], [], []  # kept for the window / trace collections below
for agent_i, EV_no_nan_shifted in enumerate(ls_EV_no_nan_shifted):
    model_name = ls_model_name[agent_i]
    outdir = ls_outdir[agent_i]
    fig_wind = fig_odor = None
    cnorm = None
    wind_windows = odor_windows = None
    if plot_around_wind:
        ##########################################################################################
        ######                    PLOT OBS AND ACTION AROUND WIND CHANGE                    ######
        ##########################################################################################
        wind_windows = tidx_around_wind_change(EV_no_nan_shifted, around_range=wind_around_range,
                                               stable_guard=steps(wind_stable_guard_s),
                                               min_time_from_init=min_time_from_init_s,
                                               min_stable_before=min_stable_before_s,
                                               drop_close_to_trial_init=drop_close_to_trial_init,
                                               verbose=True)
        EV_no_nan_shifted_around = wind_windows[wind_windows['eligible']]
        if EV_no_nan_shifted_around.shape[0] == 0:
            print(f"No eligible wind change instances found for {model_name}")
        else:
            if verbose:
                plt.figure()
                plt.hist(EV_no_nan_shifted_around['ep_idx'], bins=900)
                plt.title("episodes with eligible wind change instances")
                plt.show()
            trial_outcomes_wind = EV_no_nan_shifted_around.groupby('ep_idx')['outcome'].unique().value_counts()
            print(f"trial outcomes around wind change {trial_outcomes_wind}")

            null_src = EV_no_nan_shifted
            if filter_null:
                null_src = null_src[null_src['ep_idx'].isin(EV_no_nan_shifted_around['ep_idx'].unique())]
            null_df = null_baseline(null_src, kin_cols, median=median)
            subset_df = aligned_stats(EV_no_nan_shifted_around, kin_cols, color_by=color_by, median=median)
            fig_wind, ax_wind, cnorm = plot_aligned(subset_df, null_df, states, 'Time Since Wind Change (s)',
                                                    cnorm=None, cmap=cmap, color_by=color_by, fig_size=fig_size,
                                                    titles_on=titles_on, ylabl_title=ylabl_title)

    if plot_around_odor:
        ##########################################################################################
        ######                    PLOT OBS AND ACTION AROUND ODOR CONTACT                   ######
        ##########################################################################################
        EV_no_nan_around_odor_shifted = EV_no_nan_shifted.copy()
        EV_no_nan_around_odor_shifted['odor_contact_start'] = mark_odor_contact_start(EV_no_nan_around_odor_shifted)
        odor_windows = tidx_around_odor_contact(EV_no_nan_around_odor_shifted, around_range=odor_around_range,
                                                odor_guard=steps(odor_guard_s), verbose=True)
        EV_no_nan_around_odor_shifted = odor_windows[odor_windows['eligible']]
        if EV_no_nan_around_odor_shifted.shape[0] == 0:
            print(f"No eligible odor contact instances found for {model_name}")
        else:
            trial_outcomes_odor = EV_no_nan_around_odor_shifted.groupby('ep_idx')['outcome'].unique().value_counts()
            print(f"trial outcomes around odor contact {trial_outcomes_odor}")
            if verbose:
                plt.figure()
                plt.hist(EV_no_nan_around_odor_shifted['ep_idx'], bins=900)
                plt.title("episodes with eligible odor contact instances")
                plt.show()

            null_src = EV_no_nan_shifted
            if filter_null:
                null_src = null_src[null_src['ep_idx'].isin(EV_no_nan_around_odor_shifted['ep_idx'].unique())]
            null_df = null_baseline(null_src, kin_cols, median=median)
            subset_df = aligned_stats(EV_no_nan_around_odor_shifted, kin_cols, color_by=color_by, median=median)
            # same colorbar as the wind plot when there is one
            fig_odor, ax_odor, cnorm = plot_aligned(subset_df, null_df, states, 'Odor Contact (s)',
                                                    cnorm=cnorm, cmap=cmap, color_by=color_by, fig_size=fig_size,
                                                    titles_on=titles_on, ylabl_title=ylabl_title)

    if fig_wind is not None and fig_odor is not None:
        # same y limits per state for the two figures, fixed x limits
        for axis_i in range(len(states)):
            y_lim_w = ax_wind[axis_i, 0].get_ylim()
            y_lim_o = ax_odor[axis_i, 0].get_ylim()
            y_lim = [min(y_lim_w[0], y_lim_o[0]), max(y_lim_w[1], y_lim_o[1])]
            for a in (ax_wind[axis_i, 0], ax_odor[axis_i, 0]):
                a.set_ylim(y_lim)
                a.set_xlim((-plot_xlim_s, plot_xlim_s))
                a.set_xticks(np.linspace(-plot_xlim_s, plot_xlim_s, num=5))

    ls_wind_windows.append(wind_windows)
    ls_odor_windows.append(odor_windows)
    ls_cnorm.append(cnorm)

    if save:
        os.makedirs(outdir, exist_ok=True)
        for fig, tag in ((fig_wind, 'wind'), (fig_odor, 'odor')):
            if fig is None:
                continue
            for ext in ('pdf', 'png'):
                fname = f"{outdir}/{model_name}_w{window_length}_same_scale_{tag}{'_median' if median else ''}.{ext}"
                fig.savefig(fname)
                print(f"saved {fname}")
    plt.show()

## where do the traces come from: event windows in space, unrotated
Every window (+-`wind_around_s` / `odor_around_s` around the event) drawn with
`outcome_start_plots.draw_window_panel`, locations rotated back by `-rotate_by` so all plume
rotations align with 0 deg. Black squares: off plume, red stars: on plume, blue x: wind jump,
green circle: window start, purple triangle: window end, green dot: the event (plot_tidx 0),
black circle: the goal. Once with all windows, once with only the eligible ones.

In [ ]:
window_style = dict(PANEL_STYLE, point_size=2, endpoint_size=4, wind_marker_size=6,
                    wind_change_thresh=0.5)  # the dt1 column is the 0/1 wind_jump flag (10 deg rule)
window_outcomes = ('HOME', 'OOB', 'OOT')


def window_frame(windows_df):
    """Frame for draw_window_panel: one 'episode' (its ep_idx) per event window, unrotated locations."""
    return pd.DataFrame({
        'ep_idx': windows_df['event_idx'].values,       # start / end markers per window
        'episode': windows_df['ep_idx'].values,
        'outcome': windows_df['outcome'].values,
        'rotate_by': windows_df['rotate_by'].values,
        'loc_x': windows_df['loc_x_unrot'].values,
        'loc_y': windows_df['loc_y_unrot'].values,
        't_val': windows_df['time'].values,
        'odor_01': windows_df['odor_01'].values,
        'wind_angle_ground_theta_dt1': windows_df['wind_jump'].values.astype(float),
        'plot_tidx': windows_df['plot_tidx'].values,
        'eligible': windows_df['eligible'].values,
    })


def plot_window_collection(frame, title, outcomes=window_outcomes, style=window_style, col_w=5, dpi=150):
    """One panel with all windows plus one per outcome, shared limits (outcome_start_plots layout)."""
    panels = [('all outcomes', frame)] + [(o, frame[frame['outcome'] == o]) for o in outcomes
                                          if (frame['outcome'] == o).any()]
    x0, x1, y0, y1 = _limits(frame, style['goal_radius'])
    row_h = col_w * (y1 - y0) / (x1 - x0) + 1.0
    fig, axs = plt.subplots(1, len(panels), figsize=(col_w * len(panels), row_h), dpi=dpi,
                            sharex=True, sharey=True, squeeze=False)
    for ax, (name, g) in zip(axs.ravel(), panels):
        draw_window_panel(ax, g, legend=(name == 'all outcomes'), **style)
        ev = g[g['plot_tidx'] == 0]
        ax.plot(ev['loc_x'], ev['loc_y'], ls='', marker='o', ms=3, color='tab:green',
                label='event' if name == 'all outcomes' else None)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        rot = ', '.join(f"{int(k)}°:{v}" for k, v in g.drop_duplicates('ep_idx')['rotate_by'].value_counts().sort_index().items())
        ax.set_title(f"{name}: {g['ep_idx'].nunique()} windows from {g['episode'].nunique()} eps\n"
                     f"rotate_by {rot}", fontsize=8)
        ax.tick_params(labelsize=7)
    if axs[0, 0].get_legend_handles_labels()[0]:
        axs[0, 0].legend(loc='upper right', fontsize=6, markerscale=1.2)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


for agent_i in range(len(ls_EV_no_nan_shifted)):
    model_name, outdir = ls_model_name[agent_i], ls_outdir[agent_i]
    for event, windows in (('wind', ls_wind_windows[agent_i]), ('odor', ls_odor_windows[agent_i])):
        if windows is None or windows.empty:
            continue
        frame_all = window_frame(windows)
        for stage, frame in (('all', frame_all), ('eligible', frame_all[frame_all['eligible']])):
            if frame.empty:
                print(f"{event} {stage}: no windows")
                continue
            fig = plot_window_collection(frame, f"{model_name} | w{window_length} | {event} change windows, "
                                                f"{stage} ({frame['ep_idx'].nunique()}), unrotated to 0°")
            if save:
                fname = f"{outdir}/{model_name}_w{window_length}_windows_{event}_{stage}_unrotated.png"
                fig.savefig(fname, dpi=150, bbox_inches='tight')
                print(f"saved {fname}")
            plt.show()

## sensory trace collections (_preprint_observability.ipynb)
One faint trace per window: wind direction (unrotated, deg) or odor, the two kinematic quantities
and `zeta_shifted` (log axis, coloured by its own value with the colourbar of the aligned
figures) against time since the event. All windows, then eligible windows only.

In [ ]:
trace_alpha = 0.05   # faintness of individual traces
sensory_h = 1.2
on_color, off_color = config.traj_colormap['on'], config.traj_colormap['off']
sensory_cfg = {
    'xlim': (-plot_xlim_s, plot_xlim_s),
    'odor_eps_log': dict(ylabel='Odor\n(a.u.)', odor_segments=True, ylim=None),
    'wind_zeta_deg_unrot': dict(ylabel='Wind\nDirection\n(deg)', ylim=None),
    'speed_dt_abs': dict(ylabel=ylabl_title['speed_dt_abs'], ylim=None),
    'ang_vel_dt_abs': dict(ylabel=ylabl_title['ang_vel_dt_abs'], ylim=None),
    'zeta_shifted': dict(ylabel='Min. Error\nVariance\n' + r'(rad$^{-2}$)', yscale='log', use_colorline=True),
}
sensory_columns = {'wind': ['wind_zeta_deg_unrot', 'speed_dt_abs', 'ang_vel_dt_abs', 'zeta_shifted'],
                   'odor': ['odor_eps_log', 'speed_dt_abs', 'ang_vel_dt_abs', 'zeta_shifted']}


def plot_event_traces(windows_df, columns, xlabel, cnorm, cmap='inferno_r', fontsize=12):
    fig = plt.figure(figsize=(5, sensory_h * len(columns)), dpi=150)
    gs = fig.add_gridspec(len(columns), 1, hspace=0.5)
    axes = [fig.add_subplot(gs[i]) for i in range(len(columns))]
    for _, ev in windows_df.groupby('event_idx', sort=False):
        x = ev['plot_tidx'].values * DT
        for ax, col in zip(axes, columns):
            cfg = sensory_cfg[col]
            y = ev[col].values.astype(float)
            if cfg.get('odor_segments'):   # on-plume segments in the 'on' colour, off-plume in 'off'
                on = y > 0
                for j in range(len(y) - 1):
                    off = not on[j] and not on[j + 1]
                    ax.plot(x[j:j + 2], y[j:j + 2], lw=2 if off else 1, color=off_color if off else on_color,
                            alpha=trace_alpha)
            elif cfg.get('use_colorline'):
                colorline_trace(x, y, y, ax=ax, cmap=cmap, norm=cnorm, linewidth=2, alpha=trace_alpha)
            else:
                ax.plot(x, y, lw=1, color='black', alpha=trace_alpha)
    for i, (ax, col) in enumerate(zip(axes, columns)):
        cfg = sensory_cfg[col]
        vals = windows_df[col].values.astype(float)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.spines['left'].set_visible(True)
        if cfg.get('yscale'):
            ax.set_yscale(cfg['yscale'])
        if cfg.get('use_colorline') and cnorm is not None:
            ax.set_ylim(cnorm.vmin, cnorm.vmax * 10)
            ax.set_yticks(np.logspace(np.log10(cnorm.vmin), np.log10(cnorm.vmax), 3))
        elif cfg.get('ylim') is not None:
            ax.set_ylim(cfg['ylim'])
        else:  # autoscale from the data (LineCollections do not autoscale)
            lo, hi = np.nanmin(vals), np.nanpercentile(vals, 99.5)
            pad = 0.05 * (hi - lo or 1.0)
            ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlim(sensory_cfg['xlim'])
        ax.axvline(0, color='green', lw=1.2, ls='--')
        ax.tick_params(axis='both', which='both', length=3, labelsize=fontsize)
        ax.set_ylabel(cfg['ylabel'], fontsize=fontsize)
        if i == len(columns) - 1:
            ax.set_xlabel(xlabel, fontsize=fontsize)
        else:
            ax.tick_params(labelbottom=False)
    return fig


for agent_i in range(len(ls_EV_no_nan_shifted)):
    model_name, outdir = ls_model_name[agent_i], ls_outdir[agent_i]
    cnorm = ls_cnorm[agent_i]
    for event, windows, xlabel in (('wind', ls_wind_windows[agent_i], 'Time Since Wind Change (s)'),
                                   ('odor', ls_odor_windows[agent_i], 'Odor Contact (s)')):
        if windows is None or windows.empty:
            continue
        for stage, w in (('all', windows), ('eligible', windows[windows['eligible']])):
            if w.empty:
                print(f"{event} {stage}: no windows")
                continue
            fig = plot_event_traces(w, sensory_columns[event], xlabel, cnorm)
            fig.suptitle(f"{model_name} | {event} windows, {stage} ({w['event_idx'].nunique()})", fontsize=8, y=1.02)
            if save:
                for ext in ('svg', 'png'):
                    fname = f"{outdir}/{model_name}_w{window_length}_traces_{event}_{stage}.{ext}"
                    fig.savefig(fname, bbox_inches='tight', dpi=300, transparent=(ext == 'svg'), format=ext)
                    print(f"saved {fname}")
            plt.show()

## colour bar and axis titles, as a separate figure

In [ ]:
if cnorm is not None:
    fig, ax = plt.subplots(n_state, 1, figsize=fig_size, dpi=300)
    ax = np.atleast_2d(ax).reshape(-1, 1)
    cax = ax[min(1, n_state - 1), -1].inset_axes([0.9, 0.2, 0.04, 1.8])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap='inferno'), cax=cax)
    cbar.ax.set_yticks([])
    cbar.ax.set_yticklabels([])
    cbar.outline.set_visible(False)
    cbar.ax.text(0.5, 1.05, 'High', ha='center', va='center', fontsize=32)
    cbar.ax.text(0.5, -0.05, 'Low', ha='center', va='center', fontsize=32)
    cbar.set_label('Observability\n', rotation=270, fontsize=32, labelpad=32)
    cbar.ax.tick_params(labelsize=18)
    for n, state_name in enumerate(states):
        for side in ('bottom', 'left', 'top', 'right'):
            ax[n, 0].spines[side].set_linewidth(0)
        ax[n, 0].set_xticks([])
        ax[n, 0].set_yticks([])
        ax[n, 0].set_ylabel(ylabl_title[state_name], fontsize=28)
    if save:
        fname = f"{ls_outdir[-1]}/colorbar.pdf"
        fig.savefig(fname)
        print(f"saved {fname}")
    plt.show()